In [2]:
# IMPORTS the request 
import requests
import pandas as pd
import pymysql
import time

# CONFII
API_KEY = "G1HVGPE42T2IDZXF"

STOCKS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "GC",
    "SI"
    
]

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "root1",
    "database": "stock_db",
    "charset": "utf8mb4"
}

# DATA FETCHING  + AUTOMATED DATA VALIDATION FUNCTION
def fetch_stock_data(symbol):
    url = (
        "https://www.alphavantage.co/query?"
        f"function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}"
    )

    try:
        response = requests.get(url, timeout=15)
        data = response.json()

        if "Time Series (Daily)" not in data:
            print(f"❌ API error / limit hit for {symbol}")
            return None

        ts = data["Time Series (Daily)"]

        df = pd.DataFrame.from_dict(ts, orient="index")
        df.reset_index(inplace=True)

        df.columns = ["date", "open", "high", "low", "close", "volume"]
        df["symbol"] = symbol

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df.dropna(inplace=True)

        df[["open","high","low","close","volume"]] = df[
            ["open","high","low","close","volume"]
        ].astype(float)

        return df

    except Exception as e:
        print(f"❌ Fetch error for {symbol}: {e}")
        return None

# MYSQL CONNECT
connection = pymysql.connect(**DB_CONFIG)
cursor = connection.cursor()

#  TABLE CREATE
cursor.execute("""
CREATE TABLE IF NOT EXISTS stock_prices (
    symbol VARCHAR(20),
    date DATE,
    open FLOAT,
    high FLOAT,
    low FLOAT,
    close FLOAT,
    volume BIGINT,
    PRIMARY KEY (symbol, date)
)
""")
connection.commit()

# INSERT QUERY
insert_sql = """
INSERT IGNORE INTO stock_prices
(symbol, date, open, high, low, close, volume)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

# MAIN LOOP 
for stock in STOCKS:
    print(f"📥 Processing {stock} ...")

    df = fetch_stock_data(stock)

    if df is not None:
        for _, row in df.iterrows():
            cursor.execute(
                insert_sql,
                (
                    row["symbol"],
                    row["date"],
                    row["open"],
                    row["high"],
                    row["low"],
                    row["close"],
                    row["volume"]
                )
            )
        connection.commit()
        print(f"✅ {stock} data saved")

    time.sleep(12)  

# CLOSE 
cursor.close()
connection.close()
print("🎯 ALL STOCK DATA FETCHED & STORED SUCCESSFULLY")


📥 Processing AAPL ...
❌ Fetch error for AAPL: HTTPSConnectionPool(host='www.alphavantage.co', port=443): Max retries exceeded with url: /query?function=TIME_SERIES_DAILY&symbol=AAPL&apikey=G1HVGPE42T2IDZXF (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028)')))
📥 Processing MSFT ...
❌ Fetch error for MSFT: HTTPSConnectionPool(host='www.alphavantage.co', port=443): Max retries exceeded with url: /query?function=TIME_SERIES_DAILY&symbol=MSFT&apikey=G1HVGPE42T2IDZXF (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1028)')))
📥 Processing GOOGL ...
❌ Fetch error for GOOGL: HTTPSConnectionPool(host='www.alphavantage.co', port=443): Max retries exceeded with url: /query?function=TIME_SERIES_DAILY&symbol=GOOGL&apikey=G1HVGPE42T2IDZXF (Caused by SSLError(SSLCertVerifica